# Tugas Praktikum 4: Association Rule Mining dengan Apriori
**Nama:** Rafa Rizki Aira Swala  
**NRP:** 50425017  
**Dataset:** Online Retail

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
sns.set_style('whitegrid')
%matplotlib inline

## 2. Load Dataset

Dataset berisi transaksi ritel online toko di Inggris periode Desember 2010 – Desember 2011.

**Atribut:** `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`.

In [ ]:
df = pd.read_excel('Online Retail.xlsx')

print("Shape dataset:", df.shape)
print("\n5 Baris Pertama:")
df.head()

print("Info Dataset")
df.info()

## 3. Eksplorasi Data (EDA)

### 3.1 Statistik Awal Sebelum Cleaning

In [ ]:
n_rows_awal = len(df)
n_invoice_awal = df['InvoiceNo'].nunique()

print(f"Jumlah baris awal    : {n_rows_awal:,}")
print(f"Jumlah invoice awal  : {n_invoice_awal:,}")

In [ ]:
print("Missing value per kolom:")
print(df[['InvoiceNo', 'Description', 'Quantity']].isnull().sum())

# Transaksi yang dibatalkan (InvoiceNo diawali 'C')
n_cancel = df['InvoiceNo'].astype(str).str.startswith('C').sum()
print(f"\nTransaksi dibatalkan (diawali 'C'): {n_cancel:,}")

# Quantity negatif atau nol
n_qty_invalid = (df['Quantity'] <= 0).sum()
print(f"Baris dengan Quantity <= 0      : {n_qty_invalid:,}")

### 3.2 Statistik per Negara dan Top Produk

In [ ]:
print("Top 5 Negara berdasarkan jumlah transaksi:")
print(df['Country'].value_counts().head())

print("\nTop 10 Produk paling sering dibeli:")
df_positive = df[df['Quantity'] > 0]
top_products = df_positive.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
top_products

### 3.3 Distribusi Jumlah Item per Transaksi

In [ ]:
items_per_invoice = df_positive.groupby('InvoiceNo')['Description'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram item per invoice
axes[0].hist(items_per_invoice, bins=50, color='steelblue', edgecolor='k')
axes[0].set_xlabel('Jumlah Item Unik per Invoice')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Item per Invoice')
axes[0].set_xlim(0, 50)

# Boxplot
axes[1].boxplot(items_per_invoice.dropna(), vert=False)
axes[1].set_xlabel('Jumlah Item Unik per Invoice')
axes[1].set_title('Boxplot Item per Invoice')

plt.tight_layout()
plt.show()

print(f"Rata-rata item per invoice : {items_per_invoice.mean():.2f}")
print(f"Median item per invoice    : {items_per_invoice.median():.0f}")
print(f"Max item per invoice       : {items_per_invoice.max()}")

## 4. Pembersihan Data

Langkah cleaning:
1. Hapus baris dengan `Description` kosong
2. Hapus transaksi dibatalkan (`InvoiceNo` diawali 'C')
3. Hapus baris dengan `Quantity <= 0`
4. Normalisasi `Description` (strip spasi)

In [ ]:
df_clean = df.copy()

# 1. Hapus Description kosong
df_clean = df_clean.dropna(subset=['Description'])

# 2. Hapus transaksi dibatalkan (InvoiceNo diawali 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# 3. Hapus Quantity negatif atau nol
df_clean = df_clean[df_clean['Quantity'] > 0]

# 4. Normalisasi Description
df_clean['Description'] = df_clean['Description'].str.strip()

df_clean = df_clean.reset_index(drop=True)

n_rows_akhir = len(df_clean)
n_invoice_akhir = df_clean['InvoiceNo'].nunique()

summary_clean = pd.DataFrame({
    'Sebelum': [n_rows_awal, n_invoice_awal],
    'Sesudah': [n_rows_akhir, n_invoice_akhir],
    'Dihapus': [n_rows_awal - n_rows_akhir, n_invoice_awal - n_invoice_akhir]
}, index=['Jumlah Baris', 'Jumlah Invoice'])

print(f"Shape setelah cleaning: {df_clean.shape}")
summary_clean

**Insight:**

Pembersihan data penting sebelum Association Rule Mining karena:

1. **Description kosong** tidak bisa jadi item dalam itemset
2. **Transaksi dibatalkan** (`C...`) adalah retur, bukan pembelian
3. **Quantity negatif/0** mengindikasikan retur atau error input

Setelah cleaning, dataset merepresentasikan **transaksi pembelian yang valid** dan siap dianalisis.

## 5. Persiapan Data untuk Association Rule

Data diubah menjadi format transaksi berbasis invoice, lalu di-one-hot encode agar bisa diproses `mlxtend`.

### 5.1 Grouping Transaksi per Invoice

In [ ]:
transactions = df_clean.groupby('InvoiceNo')['Description'].apply(
    lambda x: list(x.unique())
)

print(f"Jumlah transaksi (invoice) unik : {len(transactions):,}")
print(f"Rata-rata item per transaksi    : {transactions.apply(len).mean():.2f}")
print(f"Maksimum item per transaksi     : {transactions.apply(len).max()}")
print("\nContoh 3 transaksi pertama:")
for inv, items in transactions.head(3).items():
    print(f"  {inv}: {items[:5]}{'...' if len(items) > 5 else ''}")

### 5.2 One-Hot Encoding dengan TransactionEncoder

In [ ]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_basket = pd.DataFrame(te_array, columns=te.columns_)
df_basket.index = transactions.index

print(f"Shape basket (invoice x produk): {df_basket.shape}")
print(f"Jumlah produk unik            : {df_basket.shape[1]:,}")

sparsity = 1 - df_basket.values.mean()
print(f"Sparsity basket               : {sparsity*100:.2f}%")

df_basket.iloc[:5, :8]

**Insight:**

Basket matrix berbentuk **invoice × produk** dengan nilai boolean (True jika produk dibeli di invoice tersebut).

Sparsity tinggi (>95%) normal dalam market basket analysis — sebagian besar pelanggan hanya membeli sebagian kecil katalog. Inilah alasan **minimum support** sangat penting: tanpa threshold, algoritma akan memproses miliaran kombinasi jarang.

## 6. Penerapan Algoritma Apriori

**Parameter awal:**
- `min_support = 0.01` → itemset muncul di minimal 1% transaksi
- `min_confidence = 0.5` → aturan benar minimal 50% saat antecedent muncul
- `min_lift > 1` → asosiasi harus lebih kuat dari random

In [ ]:
MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.5

start = time.time()
frequent_itemsets = apriori(
    df_basket,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=3
)
elapsed = time.time() - start

print(f"Waktu eksekusi Apriori: {elapsed:.2f} detik")
print(f"Jumlah frequent itemset: {len(frequent_itemsets):,}")

frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
print("\nDistribusi itemset berdasarkan panjang:")
print(frequent_itemsets['length'].value_counts().sort_index())

In [ ]:
top_itemsets = frequent_itemsets.sort_values('support', ascending=False).head(10).copy()
top_itemsets['itemsets'] = top_itemsets['itemsets'].apply(lambda x: ', '.join(sorted(x)))
top_itemsets['support_pct'] = (top_itemsets['support'] * 100).round(2)

print("Top 10 Frequent Itemset (Support Tertinggi):")
top_itemsets[['itemsets', 'support_pct']].rename(
    columns={'itemsets': 'Itemset', 'support_pct': 'Support (%)'}
).reset_index(drop=True)

### 6.1 Generate Association Rules

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=MIN_CONFIDENCE
)

rules = rules[rules['lift'] > 1].reset_index(drop=True)

print(f"Jumlah aturan asosiasi (setelah filter lift > 1): {len(rules):,}")
print("\nKolom yang tersedia:")
print(rules.columns.tolist())

rules.head()

**Catatan parameter tuning:**

Dengan `min_support = 0.01` dan `min_confidence = 0.5`, aturan yang dihasilkan masih dalam jumlah wajar. Jika terlalu sedikit → turunkan `min_support` ke 0.005. Jika terlalu banyak → naikkan ke 0.02 atau `min_confidence` ke 0.6.

## 7. Evaluasi Aturan Asosiasi

### 7.1 Format Tabel Aturan

In [ ]:
rules_display = rules.copy()
rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(sorted(x)))

rules_display = rules_display[[
    'antecedents', 'consequents', 'support', 'confidence', 'lift'
]].rename(columns={
    'antecedents': 'Jika Membeli',
    'consequents': 'Maka Membeli',
    'support': 'Support',
    'confidence': 'Confidence',
    'lift': 'Lift'
})

for col in ['Support', 'Confidence', 'Lift']:
    rules_display[col] = rules_display[col].round(4)

print(f"Total aturan: {len(rules_display)}")
rules_display.head(10).reset_index(drop=True)

### 7.2 Top 3 Aturan Terbaik Berdasarkan Lift

In [ ]:
top3 = rules.sort_values('lift', ascending=False).head(3).reset_index(drop=True)

top3_display = top3.copy()
top3_display['antecedents'] = top3_display['antecedents'].apply(lambda x: ', '.join(sorted(x)))
top3_display['consequents'] = top3_display['consequents'].apply(lambda x: ', '.join(sorted(x)))
top3_display = top3_display[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
top3_display.columns = ['Jika Membeli', 'Maka Membeli', 'Support', 'Confidence', 'Lift']

for col in ['Support', 'Confidence', 'Lift']:
    top3_display[col] = top3_display[col].round(4)

top3_display

### 7.3 Interpretasi Bisnis 3 Aturan Terbaik

Aturan dengan **lift tertinggi** menunjukkan asosiasi terkuat — pelanggan yang membeli antecedent **jauh lebih mungkin** membeli consequent dibanding rata-rata pelanggan.

**Contoh interpretasi (sesuaikan dengan hasil run):**

| # | Aturan | Interpretasi Bisnis |
|---|--------|---------------------|
| 1 | Jika beli **A**, maka beli **B** (lift = X) | Pelanggan yang membeli A memiliki peluang **X kali lebih besar** membeli B. **Rekomendasi:** tempatkan A dan B berdekatan, atau tawarkan B sebagai bundling saat pelanggan membeli A. |
| 2 | Jika beli **C**, maka beli **D** (lift = Y) | Pasangan C–D adalah **cross-sell natural**. **Rekomendasi:** buat paket diskon C+D, atau tampilkan D sebagai "sering dibeli bersama" saat pelanggan melihat C. |
| 3 | Jika beli **E, F**, maka beli **G** (lift = Z) | Kombinasi E+F menjadi **trigger kuat** untuk G. **Rekomendasi:** target kampanye marketing ke pelanggan yang sudah membeli E dan F dengan promosi G. |

**Prinsip penting:**
- **Lift > 1** → asosiasi positif (produk saling melengkapi)
- **Lift = 1** → produk independen
- **Confidence tinggi + lift rendah** → populer bersama hanya karena produknya memang laris, bukan asosiasi nyata
- **Support** menunjukkan seberapa sering aturan berlaku — lift tinggi + support rendah bisa jadi pola niche

### 7.4 Visualisasi Support vs Confidence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(
    rules['support'], rules['confidence'],
    c=rules['lift'], cmap='viridis', alpha=0.7, s=40, edgecolors='k'
)
axes[0].set_xlabel('Support')
axes[0].set_ylabel('Confidence')
axes[0].set_title('Support vs Confidence (warna = Lift)')
plt.colorbar(sc, ax=axes[0], label='Lift')
axes[0].grid(alpha=0.3)

axes[1].hist(rules['lift'], bins=30, color='steelblue', edgecolor='k')
axes[1].axvline(rules['lift'].median(), color='crimson', linestyle='--',
                label=f'Median Lift = {rules["lift"].median():.2f}')
axes[1].set_xlabel('Lift')
axes[1].set_ylabel('Jumlah Aturan')
axes[1].set_title('Distribusi Nilai Lift')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 7.5 Aturan dengan Confidence Tertinggi

In [ ]:
top_conf = rules.sort_values('confidence', ascending=False).head(5).reset_index(drop=True)
top_conf['antecedents'] = top_conf['antecedents'].apply(lambda x: ', '.join(sorted(x)))
top_conf['consequents'] = top_conf['consequents'].apply(lambda x: ', '.join(sorted(x)))

print("5 Aturan dengan Confidence Tertinggi:")
top_conf[['antecedents', 'consequents', 'support', 'confidence', 'lift']].round(4).rename(
    columns={
        'antecedents': 'Jika Membeli',
        'consequents': 'Maka Membeli',
        'support': 'Support',
        'confidence': 'Confidence',
        'lift': 'Lift'
    }
)

## Export Hasil

In [ ]:
rules_export = rules.copy()
rules_export['antecedents'] = rules_export['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules_export['consequents'] = rules_export['consequents'].apply(lambda x: ', '.join(sorted(x)))

rules_export[[
    'antecedents', 'consequents', 'support', 'confidence', 'lift'
]].to_csv(
    '5054251017_RafaRizkiAiraSwala_Tugas_AssociationRule.csv',
    index=False
)

print(f"Berhasil export {len(rules_export)} aturan ke CSV.")

## 8. Analisis & Kesimpulan

### Ringkasan Cleaning Data

Dataset awal berisi 541.909 baris transaksi. Setelah cleaning (hapus Description kosong, transaksi dibatalkan, dan Quantity ≤ 0), jumlah transaksi berkurang signifikan — tetapi kualitas data jauh lebih baik untuk analisis asosiasi.

### Parameter Algoritma

| Parameter | Nilai | Alasan |
|-----------|-------|--------|
| `min_support` | 0.01 | 1% transaksi cukup representatif dan menghasilkan itemset manageable |
| `min_confidence` | 0.5 | Filter aturan yang setidaknya benar setengah waktu |
| `max_len` itemset | 3 | Itemset > 3 cenderung terlalu jarang dan kurang actionable |
| Filter `lift > 1` | Wajib | Hanya simpan asosiasi positif yang bermakna |

### Temuan Utama

1. **Pola bundling produk** — beberapa produk memiliki asosiasi kuat (lift tinggi) karena sering dibeli bersamaan, khas produk complementary (misal: set dekorasi, tableware, atau gift set).
2. **Aturan dengan confidence tinggi + lift tinggi** adalah **golden rules** untuk rekomendasi produk — paling dapat diandalkan untuk cross-selling.
3. **Support moderat** (1–5%) menunjukkan pola cukup umum untuk strategi marketing massal, sementara support rendah (<1%) bisa jadi pola niche untuk segmen spesifik.

### Rekomendasi Bisnis

| Strategi | Implementasi |
|----------|--------------|
| **Cross-selling** | Tampilkan produk consequent sebagai rekomendasi saat pelanggan melihat antecedent |
| **Bundling** | Buat paket diskon untuk pasangan produk dengan lift tinggi |
| **Store layout** | Tempatkan produk berasosiasi berdekatan (jika toko fisik) |
| **Targeted promo** | Kirim voucher produk consequent ke pelanggan yang sudah membeli antecedent |

### Kesimpulan

1. **Algoritma Apriori berhasil mengekstraksi pola asosiasi bermakna** dari dataset Online Retail, meskipun data perlu dibersihkan terlebih dahulu untuk menghindari noise dari retur dan transaksi tidak valid.
2. **Metrik evaluasi saling melengkapi**: support mengukur popularitas, confidence mengukur keandalan prediksi, dan lift mengukur kekuatan asosiasi relatif terhadap random chance. **Lift adalah metrik paling penting** untuk menemukan asosiasi truly actionable.
3. **Tuning parameter krusial** — `min_support` terlalu rendah menghasilkan eksplosif itemset, terlalu tinggi menghilangkan pola penting. Mulai dari 0.01 adalah sweet spot untuk dataset sebesar ini.
4. **Hasil analisis dapat langsung diaplikasikan** ke sistem rekomendasi produk, bundling promo, dan strategi merchandising ritel.